# YouTube trending videos: views and engagement

## Summary

From November 14, 2017 to June 14, 2018, 161,470 YouTube trending records across the US, Canada, France, and GB show view counts from 223 to about 424.5 million. This notebook examines how views are associated with likes and comments among valid video-country records. It describes associations, not causes.

**Dependent variable:** `views` on the first recorded trending date.

**Comparison measures:** `likes` and `comment_count`. Category, tags, publishing time, and other factors are outside this notebook's scope.

## Table of contents

1. [Part 1: Import libraries](#part-1-import-libraries)
2. [Part 2: View the data](#part-2-view-the-data)
3. [Part 3: Clean and prepare the data](#part-3-clean-and-prepare-the-data)
4. [Part 4: Further data cleaning](#part-4-further-data-cleaning)
5. [Part 5: Visualizations](#part-5-visualizations)
6. [Part 6: Findings and recommendations](#part-6-findings-and-recommendations)

## Part 1: Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

## Part 2: View the data

Load the CSV, check its dimensions and column types, and preview a few records.

#### 1. Load the CSV file

In [ ]:
youtube_df = pd.read_csv('data/youtube.csv')

#### 2. Check the number of rows and columns

In [ ]:
youtube_df.shape

#### 3. Preview the first five rows

In [ ]:
youtube_df.head(5)

#### 4. Check the columns and data types

In [21]:
youtube_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 161470 entries, 0 to 161469
Data columns (total 18 columns):
 #   Column                  Non-Null Count   Dtype
---  ------                  --------------   -----
 0   index                   161470 non-null  int64
 1   video_id                161470 non-null  str  
 2   trending_date           161470 non-null  str  
 3   title                   161470 non-null  str  
 4   channel_title           161470 non-null  str  
 5   category_id             161470 non-null  int64
 6   publish_date            161470 non-null  str  
 7   time_frame              161470 non-null  str  
 8   published_day_of_week   161470 non-null  str  
 9   publish_country         161470 non-null  str  
 10  tags                    161470 non-null  str  
 11  views                   161470 non-null  int64
 12  likes                   161470 non-null  int64
 13  dislikes                161470 non-null  int64
 14  comment_count           161470 non-null  int64
 15  comments_di

#### 5. View each column data type

In [ ]:
youtube_df.dtypes

## Part 3: Clean and prepare the data

Create a working copy. Check missing and blank values, invalid video IDs, status flags, and full-row duplicates. `#NAME?` is a text placeholder, not a null value. Use Missingno to view missingness.

#### 1. Count missing values in each column

In [ ]:
youtube_df.isnull().sum()

#### 2. Check the percentage of missing values

In [ ]:
(youtube_df.isnull().mean() * 100).round(2)

#### 3. Check duplicate rows

In [ ]:
print('Full-row duplicates:', youtube_df.duplicated().sum())
print('Duplicates ignoring index:', youtube_df.drop(columns='index').duplicated().sum())

#### 4. Count unusable video IDs

In [ ]:
(youtube_df['video_id'] == '#NAME?').sum()

#### 5. Count rows with comments or ratings disabled

In [ ]:
print('Comments disabled:', youtube_df['comments_disabled'].sum())
print('Ratings disabled:', youtube_df['ratings_disabled'].sum())

#### 6. Visualize missing values

In [ ]:
msno.matrix(youtube_df)
plt.show()

## Part 4: Further data cleaning

Create `newyoutube_df`, remove invalid IDs and repeated rows, convert the dates, and keep the first trending record for each video-country pair.

#### 1. Create a clean working dataframe and remove the row index

In [ ]:
newyoutube_df = youtube_df.drop(columns='index').drop_duplicates().copy()
newyoutube_df.shape

#### 2. Remove rows with an unusable video ID

In [ ]:
newyoutube_df = newyoutube_df[newyoutube_df['video_id'] != '#NAME?'].copy()
newyoutube_df.shape

#### 3. Convert the date columns to datetime

In [ ]:
newyoutube_df['trending_date'] = pd.to_datetime(newyoutube_df['trending_date'], format='%y.%d.%m')
newyoutube_df['publish_date'] = pd.to_datetime(newyoutube_df['publish_date'], format='%d/%m/%Y')
newyoutube_df[['trending_date', 'publish_date']].dtypes

#### 4. Keep the first trending record for each video in each country

In [ ]:
newyoutube_df = newyoutube_df.sort_values('trending_date', kind='stable')
newyoutube_df = newyoutube_df.drop_duplicates(subset=['video_id', 'publish_country'], keep='first')
newyoutube_df.shape

#### 5. Prepare the likes and comments data

In [ ]:
likes_df = newyoutube_df[newyoutube_df['ratings_disabled'] == False]
comments_df = newyoutube_df[newyoutube_df['comments_disabled'] == False]
print('Rows for likes:', len(likes_df))
print('Rows for comments:', len(comments_df))

## Part 5: Visualizations

Compare views with likes and comments using Spearman correlation and simple charts. Skip records where the corresponding feature is disabled. The log-scale scatterplots leave out zero counts.

#### 1. Check the relationship between views and likes

In [ ]:
likes_df['views'].corr(likes_df['likes'], method='spearman')

#### 2. Check the relationship between views and comments

In [ ]:
comments_df['views'].corr(comments_df['comment_count'], method='spearman')

#### 3. Plot the distribution of views

In [ ]:
sns.histplot(np.log1p(newyoutube_df['views']), bins=40)
plt.title('Distribution of views')
plt.xlabel('log(1 + views)')
plt.show()

#### 4. Plot views and likes

In [ ]:
sns.scatterplot(data=likes_df[likes_df['likes'] > 0], x='views', y='likes', alpha=0.3)
plt.xscale('log')
plt.yscale('log')
plt.title('Views and likes')
plt.show()

#### 5. Plot views and comments

In [ ]:
sns.scatterplot(data=comments_df[comments_df['comment_count'] > 0], x='views', y='comment_count', alpha=0.3)
plt.xscale('log')
plt.yscale('log')
plt.title('Views and comments')
plt.show()

## Part 6: Findings and recommendations

### Findings

- Views and likes: report the direction and strength of the observed association.
- Views and comments: report the direction and strength of the observed association.

### Limitations

Note invalid IDs, repeated records, disabled engagement fields, and why the results describe associations rather than causes.

### Recommendations

Suggest only next steps supported by the results. Do not claim that likes or comments cause views to increase.